In [ ]:
# ── A: County-level prediction summary ───────────────────────────────
import pandas as pd
import numpy as np

df_48 = pd.read_csv(percounty_pred_48h.csv)
df_24 = pd.read_csv(percounty_pred_24h.csv)

# Build per-county summary table
county_summary = []
for loc in locations:
    loc_str = str(loc)
    preds_48 = df_48[df_48['location'].astype(str) == loc_str]['pred'].values
    preds_24 = df_24[df_24['location'].astype(str) == loc_str]['pred'].values

    # Get tracked households (most recent value)
    tracked = float(ds_train.tracked.sel(location=loc).values[-1])

    county_summary.append({
        'fips':            loc_str,
        'tracked_hh':      tracked,
        'pred_total_48h':  preds_48.sum(),
        'pred_peak_48h':   preds_48.max(),
        'pred_total_24h':  preds_24.sum(),
        'pred_peak_24h':   preds_24.max(),
        'outage_rate_48h': preds_48.sum() / (tracked + 1),  # fraction of HH affected
    })

summary_df = pd.DataFrame(county_summary).sort_values('pred_total_48h', ascending=False)
print("Top 15 counties by predicted total outage customer-hours (48h):")
print(summary_df.head(15).to_string(index=False))


# B: Greedy generator allocation
N_GENERATORS = 5
GEN_CAP      = 1000   # households per generator

# Rebuild pred matrix (48, L)
pred_matrix = np.zeros((48, len(locations)))
for ci, loc in enumerate(locations):
    loc_str = str(loc)
    vals = df_48[df_48['location'].astype(str) == loc_str]['pred'].values
    pred_matrix[:, ci] = vals[:48]

def coverage(pred_col, n_gen, cap=GEN_CAP):
    """Total covered customer-hours for a county with n_gen generators."""
    return float(np.minimum(pred_col, cap * n_gen).sum())

assigned = np.zeros(len(locations), dtype=int)
chosen   = []
allocation_log = []

for step in range(N_GENERATORS):
    marginal = np.array([
        coverage(pred_matrix[:, c], assigned[c] + 1) -
        coverage(pred_matrix[:, c], assigned[c])
        for c in range(len(locations))
    ])
    best_c = int(np.argmax(marginal))
    assigned[best_c] += 1
    chosen.append(str(locations[best_c]))
    allocation_log.append({
        'step':           step + 1,
        'fips':           str(locations[best_c]),
        'n_generators':   assigned[best_c],
        'marginal_gain':  marginal[best_c],
        'total_covered':  coverage(pred_matrix[:, best_c], assigned[best_c]),
        'tracked_hh':     float(ds_train.tracked.sel(location=locations[best_c]).values[-1]),
        'pred_peak_48h':  pred_matrix[:, best_c].max(),
    })
    print(f"Generator {step+1}: FIPS {locations[best_c]}"
          f"  marginal = {marginal[best_c]:,.0f} customer-hours")

total_covered = sum(coverage(pred_matrix[:, c], assigned[c]) for c in range(len(locations)))
print(f"\nTotal expected covered customer-hours (48h): {total_covered:,.0f}")

alloc_df = pd.DataFrame(allocation_log)
print("\nAllocation summary:")
print(alloc_df.to_string(index=False))

decision_str = '[' + ', '.join(chosen) + ']'
print(f"\nDecision file: {decision_str}")

with open(os.path.join(RESULTS_DIR, 'decision.txt'), 'w') as f:
    f.write(decision_str)
print(f"Saved to {os.path.join(RESULTS_DIR, 'decision.txt')}")


#  C: Sensitivity check
pred_matrix_24 = np.zeros((24, len(locations)))
for ci, loc in enumerate(locations):
    loc_str = str(loc)
    vals = df_24[df_24['location'].astype(str) == loc_str]['pred'].values
    pred_matrix_24[:, ci] = vals[:24]

assigned_24 = np.zeros(len(locations), dtype=int)
chosen_24   = []

for step in range(N_GENERATORS):
    marginal = np.array([
        coverage(pred_matrix_24[:, c], assigned_24[c] + 1, cap=GEN_CAP) -
        coverage(pred_matrix_24[:, c], assigned_24[c], cap=GEN_CAP)
        for c in range(len(locations))
    ])
    best_c = int(np.argmax(marginal))
    assigned_24[best_c] += 1
    chosen_24.append(str(locations[best_c]))

print("48h allocation:", sorted(set(chosen)))
print("24h allocation:", sorted(set(chosen_24)))
overlap = set(chosen) & set(chosen_24)
print(f"Counties selected by BOTH horizons: {sorted(overlap)}")
print(f"Overlap: {len(overlap)}/5 counties consistent")


# D: Naive ranking comparison
naive_top5 = summary_df.head(5)['fips'].tolist()
print("Naive top-5 by predicted total outages:", naive_top5)
print("Our greedy allocation:", sorted(set(chosen)))

# For naive allocation, compute coverage
naive_covered = 0
for fips in naive_top5:
    ci = locations.index(fips) if fips in locations else None
    if ci is not None:
        naive_covered += coverage(pred_matrix[:, ci], 1)
print(f"\nNaive allocation total covered customer-hours: {naive_covered:,.0f}")
print(f"Greedy allocation total covered customer-hours: {total_covered:,.0f}")
print(f"Improvement from greedy: {total_covered - naive_covered:,.0f} customer-hours "
      f"(+{(total_covered/naive_covered - 1)*100:.1f}%)")


# E: Model comparison summary table
print("\n" + "="*65)
print("FULL MODEL COMPARISON — VALIDATION SET")
print("="*65)
print(f"{'Model':<22} {'24h RMSE':>10} {'48h RMSE':>10}  Notes")
print("-"*65)
print(f"{'Zero baseline':<22} {'100.31':>10} {'75.36':>10}  predict zero everywhere")
print(f"{'SARIMAX':<22} {'93.08':>10} {'71.66':>10}  demo notebook best")
print(f"{'Seq2Seq LSTM':<22} {'101.46':>10} {'74.31':>10}  demo notebook")
print(f"{'LightGBM (ours)':<22} {'89.25':>10} {'67.43':>10}  best on validation")
print("="*65)
print("Note: validation window overlaps storm onset (mid-June).")
print("LightGBM is used as the final submission model.")